# 06 · 고정 모델 분석과 열처리 전후 추적

현재 연구 모델을 고정하여 사용하는 화면입니다. **새 학습은 실행하지 않습니다.**

1. 위 메뉴 **Run → Run All Cells**로 입력 화면과 저장된 결과를 표시합니다.
2. 새 영상을 분석하려면 경로를 입력하고 **① 전 영상 분석 → ② 후 영상 분석 → ③ 전후 정렬·추적** 순서로 누릅니다.
3. 분석 결과는 데이터셋 폴더 아래 별도 실행 폴더에 CSV·JSON·PNG로 저장됩니다.

제공 좌표 모드는 좌표 CSV의 위치만 사용하며 기존 정답 라벨을 모델에 넣지 않습니다.
좌표가 없는 영상에는 밝기 대비 후보 모드를 사용할 수 있지만 실제 결함 전체를 검출하는 모델은 아닙니다.
종류별 수는 후보 수이며, 전후 종류 변화는 **물리적 전환으로 확정되지 않은 후보**입니다.
단일 uint8 RGB 영상 전용이고 3D·세부형 a–f·실제 길이 단위 분석은 아직 포함하지 않습니다.


In [ ]:
import json
from pathlib import Path

from sic_xrt_analyzer.analysis.notebook import build_workbench, show_result
from sic_xrt_analyzer.registration.research import load_analysis

workbench = build_workbench("research_analysis.local.json")

## 최근 완료 결과

아래는 저장된 결과를 다시 보여줍니다. 전체 실행해도 분석·학습을 반복하지 않습니다.
새 분석을 하려면 위 버튼을 누르세요. 흰 표시점은 낮은 모델 점수입니다.

In [ ]:
settings_path = Path("research_analysis.local.json")
if settings_path.exists():
    settings = json.loads(settings_path.read_text(encoding="utf-8"))
    for key in ("before_result", "after_result", "tracking_result"):
        result_file = settings.get(key)
        if result_file and Path(result_file).is_file():
            show_result(load_analysis(result_file))
else:
    print("로컬 설정을 입력한 뒤 위 분석 버튼을 사용하세요.")

## 결과를 읽는 법

- `predictions.csv`: 원본 좌표, 예측 종류, 종류별 점수. 원본 라벨을 덮어쓰지 않습니다.
- `excluded_points.csv`: 영상 경계, 중복 좌표 등으로 분류하지 않은 항목.
- `matches.csv`: 전후 연결 후보. `tentative_match`만 잠정 연결이고 `ambiguous`는 복수 상대가 있는 점입니다.
- `unmatched`는 연결하지 못했다는 뜻이며 결함 생성·소멸을 뜻하지 않습니다.
- 정렬 실패 시 결과에 이유가 표시되고 연결을 만들지 않습니다.
- 수동 대응점은 원본 좌표 `before_x,before_y,after_x,after_y` CSV로 최소 6쌍을 영상에 분산해 지정합니다.

이 화면의 추론·추적 결과는 전문가가 확정한 정답 데이터와 구분해야 합니다.